# Nonlinear timing: typed inference, coordinate charts, joint NUTS

This prototype walks the **new** nonlinear-timing API introduced by the
timing-coordinate-charts feature. Compared with the older `sample=` / `transform=`
switches it makes three things explicit:

1. **A typed inference plan** (`TimingInference`) — you name what is *marginalized*;
   every unmentioned timing axis is *sampled*. Each fitpar gets exactly one
   disposition: `sample`, `marginalize_delta_flat`, or `marginalize_z_prior`.
2. **A coordinate chart per sampled axis** — `affine_normal` for a Gaussian delta
   prior (globally affine in the sampler coordinate) or `prior_pit` for a
   bounded/non-Gaussian prior (a local PIT chart).
3. **One affine layer** — `whitening=None` is the identity static layer (sampler
   coordinate `z`), the only legal setting for the dynamic joint transport used
   by full-basis NUTS.

Everything is simulated with PINT, so no external data is needed. Run top to
bottom in the MetaPulsar devcontainer.


In [1]:
import os
os.environ.setdefault("JAX_ENABLE_X64", "1")

import tempfile
from pathlib import Path

import jax
import numpy as np

import discovery as ds
from metapulsar import create_metapulsar
from metapulsar.sandbox_tempo2 import configure_logging
from nltiming import NonLinearTimingModel, TimingInference, WhiteningConfig
from nltiming.sampling import numpyro as N

ds.config(kernels="metamath")
configure_logging(level="WARNING")
workdir = Path(tempfile.mkdtemp(prefix="nlt_charts_"))


## 1. Simulate a demo pulsar

~150 TOAs from PINT's bundled `NGC6440E` example par file. Because the TOAs are
drawn from the model itself (plus white noise) we know the truth exactly.


In [2]:
import pint.config
from pint.models import get_model
from pint.simulation import make_fake_toas_uniform

np.random.seed(42)
model = get_model(pint.config.examplefile("NGC6440E.par"))
toas = make_fake_toas_uniform(
    startMJD=53400, endMJD=56000, ntoas=150, model=model, obs="gbt",
    error=1.0, add_noise=True,
)
par_path, tim_path = workdir / "demo.par", workdir / "demo.tim"
par_path.write_text(model.as_parfile())
toas.write_TOA_file(str(tim_path), format="tempo2")

mp = create_metapulsar(
    {"demo": [{"par": str(par_path), "tim": str(tim_path), "timing_package": "pint"}]},
    use_pulse_numbers="no",
)
print("pulsar :", mp.name, " TOAs:", len(mp.toas))
print("fitpars:", list(mp.fitpars))


pulsar : J1748-2021  TOAs: 150
fitpars: ['RAJ', 'DECJ', 'F0', 'F1', 'DM', 'DM1', 'DM2', 'Offset_demo']


## 2. The typed inference plan

There is no `sample=` any more. You choose a `TimingInference`:

- `TimingInference.sample_all()` — every timing axis is sampled (the joint
  full-basis model requires this).
- `TimingInference.default()` — a versioned preset marginalizes the linear
  spindown/dispersion/jump nuisances (delta-flat) and samples the rest.
- `TimingInference.groups(delta_flat=[...], z_prior=[...])` — name exactly which
  axes are marginalized, and how; everything else is sampled.

`delta_flat` marginalization is the improper flat-in-delta GP; `z_prior`
marginalization is a *proper* unit-normal GP on the whitened coefficient — a
different measure with a different fingerprint.


In [3]:
for label, inf in [
    ("default()", TimingInference.default()),
    ("groups(z_prior=[DM], delta_flat=[DM1, DM2])",
     TimingInference.groups(z_prior=["DM"], delta_flat=["DM1", "DM2"])),
    ("sample_all()", TimingInference.sample_all()),
]:
    c = NonLinearTimingModel(engines="jug", inference=inf, name="timing").for_pulsar(mp)
    print(f"{label:48s} sampled={c.sampled}")
    print(f"{'':48s} marg_delta={c.plan.marginalized_delta} marg_z={c.plan.marginalized_z}")


[!] Unrecognized par file parameters (ignored by JUG): DMDATA, NE_SW1
[FITTER] noise_config was None, auto-detecting from par file


/workspaces/metapulsar/ref-packages/nltiming/src/nltiming/nonlinear_timing_model.py:1304: LocallyMarginalizedTimingWarning: analytically marginalized timing axes are not certified identically linear, so their integration is local: ['RAJ', 'DECJ', 'F0', 'F1']. The plan is honored; pass coordinate_policy with nonidentically_linear_marginalization='ignore' to silence.
  base = self._unconditioned_for_pulsar(pulsar, engine)


default()                                        sampled=()
                                                 marg_delta=('RAJ', 'DECJ', 'F0', 'F1', 'DM', 'DM1', 'DM2', 'Offset_demo') marg_z=()
groups(z_prior=[DM], delta_flat=[DM1, DM2])      sampled=('RAJ', 'DECJ', 'F0', 'F1', 'Offset_demo')
                                                 marg_delta=('DM1', 'DM2') marg_z=('DM',)
sample_all()                                     sampled=('RAJ', 'DECJ', 'F0', 'F1', 'DM', 'DM1', 'DM2', 'Offset_demo')
                                                 marg_delta=() marg_z=()


## 3. One chart per sampled axis

`ctx.chart_summary()` reports, for every proper (sampled or z-prior) axis, its
disposition, its `delta -> z` chart, whether the engine/registry certifies it
identically linear, and the resolved prior. Gaussian-delta (identically linear)
axes get an `affine_normal` chart; bounded cheat-priors get a local `prior_pit`
chart.


In [4]:
ctx = NonLinearTimingModel(
    engines="jug", inference=TimingInference.sample_all(), name="timing"
).for_pulsar(mp)

print(f"{'axis':14s} {'disposition':14s} {'chart':14s} {'lin?':5s} prior")
for d in ctx.chart_summary():
    print(f"{d['name']:14s} {d['disposition']:14s} {d['chart']:14s} "
          f"{str(d['identically_linear']):5s} {d['prior_family']} ({d['prior_source']})")
print("\nidentically linear:", ctx.identically_linear)


axis           disposition    chart          lin?  prior
RAJ            sample         prior_pit      False uniform (cheat_wls)
DECJ           sample         prior_pit      False uniform (cheat_wls)
F0             sample         prior_pit      False uniform (cheat_wls)
F1             sample         prior_pit      False uniform (cheat_wls)
DM             sample         affine_normal  True  normal (cheat_wls)
DM1            sample         affine_normal  True  normal (cheat_wls)
DM2            sample         affine_normal  True  normal (cheat_wls)
Offset_demo    sample         affine_normal  True  normal (cheat_wls)

identically linear: ('DM', 'DM1', 'DM2', 'Offset_demo')


## 4. delta-flat vs z-prior are distinct records

The same declared subset marginalized two different ways produces two different
plan fingerprints — the likelihood normalization differs (improper flat vs proper
unit-normal), so run products can never be confused.


In [5]:
subset = ["DM"]
ctx_df = NonLinearTimingModel(
    engines="jug", inference=TimingInference.groups(delta_flat=subset), name="timing"
).for_pulsar(mp)
ctx_zp = NonLinearTimingModel(
    engines="jug", inference=TimingInference.groups(z_prior=subset), name="timing"
).for_pulsar(mp)
print("delta-flat plan:", ctx_df.plan.fingerprint()[:24])
print("z-prior    plan:", ctx_zp.plan.fingerprint()[:24])
print("distinct        :", ctx_df.plan.fingerprint() != ctx_zp.plan.fingerprint())


delta-flat plan: sha256:167bbaec6514c9917
z-prior    plan: sha256:488f67a2e76809955
distinct        : True


## 5. Joint full-basis model + block-mass NUTS

`ctx.discovery_signals(joint=True)` emits the joint timing signals; `N.joint_model`
maps one standard-normal `xi` through the dynamic transport to `(z, GP coeffs)`.
`N.nuts` now defaults to `dense_mass="auto"`: with two-or-more red-noise
hyperparameters it adapts a dense block over `model.hyper_sites` only, leaving the
intended-white `xi` on an identity mass. We fix the noise here for a quick demo.


In [6]:
nd = {f"{mp.name}_efac": 1.0, f"{mp.name}_log10_t2equad": -8.0}
psl = ds.PulsarLikelihood([
    mp.residuals,
    ds.makenoise_measurement_simple(mp, nd),
    ds.makegp_fourier(mp, ds.powerlaw, 15, name="rednoise"),
    *ctx.discovery_signals(joint=True),
])
# Fix the red-noise hyper for a fast timing-only demo (leave them free for science).
fixed = {**nd,
         f"{mp.name}_rednoise_log10_A": -14.0,
         f"{mp.name}_rednoise_gamma": 3.0}
jm = N.joint_model(psl, ctx, fixed=fixed)
print("xi site:", jm.xi_site, " dim:", jm.transport.dimension, " hyper:", jm.hyper_sites)

mcmc = N.nuts(jm, ctx, num_warmup=500, num_samples=1000, progress_bar=True)
mcmc.run(jax.random.PRNGKey(0), extra_fields=N.NUTS_EXTRA_FIELDS)


xi site: J1748-2021_timing_joint_xi  dim: 38  hyper: ()


sample: 100%|██████████| 1500/1500 [00:09<00:00, 150.66it/s, 127 steps of size 3.03e-02. acc. prob=0.71]


### Decode to physical timing parameters

The dynamic transport is a *sampling* reparameterization, so the raw latent is
not decodable on its own — decode at sample time with `jm.to_df`, which applies
the transport and the chart to every draw.


In [7]:
df = jm.to_df(mcmc.get_samples())
for name in ("F0", "F1"):
    col = np.asarray(df[f"{ctx.name_stem}_{name}_theta_native"])
    truth = float(getattr(model, name).value)
    lo, hi = np.percentile(col, [16, 84])
    print(f"{name}: truth={truth:.12g}  68%=[{lo:.12g}, {hi:.12g}]")


F0: truth=61.485476554  68%=[61.4854765524, 61.4854765537]
F1: truth=-1.181e-15  68%=[-1.18394331227e-15, -1.1745253638e-15]


## 6. Chain-preserving diagnostics

`N.chain_diagnostics` returns `group_by_chain=True` samples plus the recorded
integrator fields and the tree-depth saturation fraction — chains are never
pooled before diagnosis. (`mcmc.run(..., extra_fields=N.NUTS_EXTRA_FIELDS)` above
made the fields available.)


In [8]:
diag = N.chain_diagnostics(mcmc, max_tree_depth=10)
print("num_steps shape (chains, draws):", diag["num_steps"].shape)
print("tree-depth saturation fraction :", round(diag["tree_depth_saturation_fraction"], 4))
print("saturated (>10%)               :", diag["tree_depth_saturated"])
print("mean accept prob               :", round(float(diag["accept_prob"].mean()), 3))


num_steps shape (chains, draws): (1, 1000)
tree-depth saturation fraction : 0.0
saturated (>10%)               : False
mean accept prob               : 0.714


## 7. The same context through Enterprise

`ntm.enterprise_signal()` returns an Enterprise-native signal for the same plan.
Pass `sample_z_coefficients=True` to promote any `z_prior` block from an
integrated GP to sampled `GPCoefficients` (unit-normal prior) — for a jointly
sampled/decentered workflow.


In [9]:
from enterprise.signals import parameter, signal_base, white_signals

ntm_zp = NonLinearTimingModel(
    engines="jug", inference=TimingInference.groups(z_prior=["DM"]),
    whitening=WhiteningConfig(), name="timing",
)
white = white_signals.MeasurementNoise(efac=parameter.Constant(1.0))
pta = signal_base.PTA([(white + ntm_zp.enterprise_signal(sample_z_coefficients=True))(mp)])
print("enterprise params:", pta.param_names)


enterprise params: ['J1748-2021_timing_x_0', 'J1748-2021_timing_x_1', 'J1748-2021_timing_x_2', 'J1748-2021_timing_x_3', 'J1748-2021_timing_x_4', 'J1748-2021_timing_x_5', 'J1748-2021_timing_x_6', 'J1748-2021_timing_zprior_coefficients_0']
